In [ ]:
# Switch path to root of project
import os
os.environ["CUDA_VISIBLE_DEVICES"]="2"

import torch
from urllib.request import urlopen

from PIL import Image
from open_clip import create_model_from_pretrained, get_tokenizer

# Load the model and config files from the Hugging Face Hub
model, preprocess = create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')


In [2]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = model.to(device)
model = model.eval()

In [ ]:
# load abdomen vqa data
import json
# vqa_train_file = "/data/xxx/hallucination/CARES/IU_Xray/training.json"
# vqa_train_file = "/data/xxx/hallucination/CARES/OmniMedVQA/training_masks_top4.json"
vqa_train_file = "/data/xxx/hallucination/PathVQA/pvqa/training_masks_top4.json"
# vqa_train_file = "/data/xxx/hallucination/VQA_RAD/data/training_masks_top4.json"
# vqa_train_file = "/data/xxx/hallucination/Slake/data/training_masks_all.json"

# vqa_train_file = "/data/xxx/hallucination/IU_Xray/data_report/training.json"
# vqa_train_file = "/data/xxx/hallucination/MIMIC_CXR/data_report/training.json"


with open(vqa_train_file, "r") as f:
    vqa_data = json.load(f)
# Check the loaded data
print(f"Loaded {len(vqa_data)} vqa data")
# filter only english data
vqa_data_train = vqa_data
print(f"Filtered {len(vqa_data_train)} vqa data")

Loaded 19755 vqa data
Filtered 19755 vqa data


In [ ]:
# sample images
import os
import random
from pathlib import Path

from sklearn.manifold import TSNE
import numpy as np

import matplotlib.pyplot as plt

# root_dir = Path("/data/xxx/hallucination/IU_Xray/iu_xray/images")
# root_dir = Path("/data/xxx/hallucination/OmniMedVQA/VQA/raw/OmniMedVQA")
root_dir = Path("/data/xxx/hallucination/PathVQA/pvqa/images/train")
# root_dir = Path("/data/xxx/hallucination/Slake/imgs")
# root_dir = Path("/data/xxx/hallucination/VQA_RAD/images/")
# image_paths = list(root_dir.rglob("source.jpg"))

# root_dir = Path("/data/xxx/hallucination/IU_Xray/iu_xray/images")
# root_dir = Path("/data/xxx/hallucination/MIMIC_CXR/sampled_files_train")

image_paths = list(set([i['image'] for i in vqa_data_train]))

# Step 2: Randomly sample 100 images
sampled_paths = image_paths

print(f"Found {len(image_paths)} training images, sampled {len(sampled_paths)}")

Found 2599 training images, sampled 2599


In [35]:
full_sampled_paths = []
for path in sampled_paths:
    full_path = os.path.join(root_dir, path)
    full_sampled_paths.append(full_path)

In [13]:
import torch.nn.functional as F
from tqdm import tqdm

In [14]:
def interpolate_attention_map(attn_map: torch.Tensor, src_H: int = 14, tgt_H: int = 24) -> torch.Tensor:
    assert attn_map.shape[0] == src_H * src_H

    # Step 1: reshape to [1, 1, H, W]
    attn_2d = attn_map.view(1, 1, src_H, src_H)

    # Step 2: interpolate to target resolution
    attn_interp = F.interpolate(attn_2d, size=(tgt_H, tgt_H), mode='bilinear', align_corners=False)

    # Step 3: flatten to [tgt_H * tgt_H]
    return attn_interp.view(tgt_H * tgt_H)

In [ ]:

attn_map_dict = {}
attn_map_dict_before_interpolate = {}
start_index, sample_number = 0, len(sampled_paths)
# start_index, sample_number = 0, 10
ind = 0
cos_matrix_dict = {}
# sample = "/data/xxx/hallucination/Slake/imgs/xmlab44/source.jpg"
for img_path in tqdm(full_sampled_paths[start_index:start_index+sample_number]):  # label: organ/lesion/other
    relative_path = sampled_paths[ind]
    ind += 1
    inputs = preprocess(Image.open(img_path)).to("cuda").unsqueeze(0)  # [1, 3, 224, 224]
    with torch.no_grad():
        image_features, attentions = model.encode_image(inputs)
        # image_features = model.encode_image(inputs)
    # print(attentions.shape, "attentions", attentions)
    cls_to_patches = attentions[-1].squeeze(0).mean(0)[0,1:]  # [N]
    attn_map_dict_before_interpolate[relative_path] = cls_to_patches.cpu().numpy()
    cls_to_patches = interpolate_attention_map(cls_to_patches, src_H=14, tgt_H=24)
    attn_map_dict[relative_path] = cls_to_patches.cpu().numpy()
    


  0%|          | 0/2599 [00:00<?, ?it/s]

100%|██████████| 2599/2599 [01:17<00:00, 33.42it/s]


In [16]:
image_features.size()

torch.Size([1, 197, 512])

In [35]:
attn_map_dict.keys()

dict_keys(['xmlab163/source.jpg', 'xmlab480/source.jpg', 'xmlab182/source.jpg', 'xmlab332/source.jpg', 'xmlab509/source.jpg', 'xmlab9/source.jpg', 'xmlab549/source.jpg', 'xmlab8/source.jpg', 'xmlab51/source.jpg', 'xmlab123/source.jpg', 'xmlab91/source.jpg', 'xmlab358/source.jpg', 'xmlab265/source.jpg', 'xmlab417/source.jpg', 'xmlab484/source.jpg', 'xmlab181/source.jpg', 'xmlab31/source.jpg', 'xmlab502/source.jpg', 'xmlab176/source.jpg', 'xmlab134/source.jpg', 'xmlab136/source.jpg', 'xmlab523/source.jpg', 'xmlab596/source.jpg', 'xmlab297/source.jpg', 'xmlab168/source.jpg', 'xmlab606/source.jpg', 'xmlab459/source.jpg', 'xmlab400/source.jpg', 'xmlab612/source.jpg', 'xmlab560/source.jpg', 'xmlab109/source.jpg', 'xmlab450/source.jpg', 'xmlab225/source.jpg', 'xmlab189/source.jpg', 'xmlab479/source.jpg', 'xmlab57/source.jpg', 'xmlab50/source.jpg', 'xmlab81/source.jpg', 'xmlab461/source.jpg', 'xmlab64/source.jpg', 'xmlab301/source.jpg', 'xmlab306/source.jpg', 'xmlab11/source.jpg', 'xmlab288/so

In [ ]:
# save cosine similarity matrix
import pickle
# save_root_path = "/data/xxx/hallucination/CARES/IU_Xray/steering_data_biomed/"
# save_root_path = "/data/xxx/hallucination/CARES/OmniMedVQA/steering_data_biomed/"
save_root_path = "/data/xxx/hallucination/PathVQA/pvqa/steering_data_biomed/"
# save_root_path = "/data/xxx/hallucination/Slake/steering/steer_data_biomed/"
# save_root_path = "/data/xxx/hallucination/VQA_RAD/steering/steer_data_biomed/"
# save_root_path = "/data/xxx/hallucination/IU_Xray/data_report/steering_data_biomed/"
# save_root_path = "/data/xxx/hallucination/MIMIC_CXR/data_report/steering_data_biomed/"


if not os.path.exists(save_root_path):
    os.makedirs(save_root_path)
# save cosine similarity matrix
with open(save_root_path + "attn_map_dict.pkl", "wb") as f:
    pickle.dump(attn_map_dict, f)